In [ ]:
!pip install easyocr pillow pandas numpy openpyxl

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import zipfile
import shutil  # Import shutil to remove directories
from PIL import Image, ImageFilter, ImageOps, ImageEnhance
import easyocr
import pandas as pd
import numpy as np
import re  # Add regex library
from concurrent.futures import ThreadPoolExecutor

# Specify the folder in Google Drive containing all category folders with zip files
main_folder = '/content/drive/MyDrive/StudyTok'

# Initialize EasyOCR reader for Arabic and English
reader = easyocr.Reader(['ar', 'en'])

def process_image(image_path):
    # Open the image and preprocess
    image = Image.open(image_path)
    image = ImageOps.grayscale(image)  # Convert to grayscale
    image = image.filter(ImageFilter.MedianFilter())  # Apply a filter to reduce noise
    image = ImageOps.invert(image)  # Invert image if text is white on black
    image = image.point(lambda p: p > 128 and 255)  # Thresholding to create a binary image

    # Increase contrast
    enhancer = ImageEnhance.Contrast(image)
    image = enhancer.enhance(2)

    # Convert the PIL image to a numpy array
    image_np = np.array(image)

    # Extract text using EasyOCR
    text = reader.readtext(image_np, detail=0)

    # Join all detected texts into one string
    text = ' '.join(text)

    # Initialize cleaned_text with the extracted text
    cleaned_text = text

    # Replace specific incorrect characters and patterns
    cleaned_text = cleaned_text.replace('@', '')
    cleaned_text = cleaned_text.replace('9', 'و')
    cleaned_text = cleaned_text.replace('9I', 'أو').replace('9l', 'أو')
    cleaned_text = cleaned_text.replace('9ل', 'لو')
    cleaned_text = cleaned_text.replace('1(', '').replace('2(', '').replace('3(', '').replace('4(', '').replace('5(', '')
    cleaned_text = cleaned_text.replace('6(', '').replace('7(', '').replace('8(', '').replace('9(', '').replace('(', '')

    # Clean unwanted words and patterns
    cleaned_text = re.sub(r'\bSTUDVTOK\b|\bstudytok\b|\studytokeg\b|\bstudytokru\b|\Studytokru\b|\Studytokeg\b|\STUDVTOKEG\b', '', cleaned_text)  # Remove variations of 'studytok'
    cleaned_text = re.sub(r'\b\d+[\\/]\d+\b', '', cleaned_text)

    # Remove extra spaces and unwanted characters
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()  # Remove multiple spaces

    # Return the filename without extension and the cleaned extracted text as a tuple
    return (os.path.splitext(os.path.basename(image_path))[0], cleaned_text)

def extract_zip_files(zip_file_path, extract_to):
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)

def process_zip_file(zip_file_path, subfolder_path):
    # Extract the zip file
    extract_to = os.path.join(subfolder_path, 'extracted_images')
    os.makedirs(extract_to, exist_ok=True)

    # Extract the zip file
    extract_zip_files(zip_file_path, extract_to)

    # Extract the zip file name (without extension) to use as the Excel file name
    zip_filename = os.path.splitext(os.path.basename(zip_file_path))[0]

    # List all extracted image files
    image_files = sorted(os.listdir(extract_to))

    # Skip the first and last image
    if len(image_files) > 2:
        image_files = image_files[1:-1]  # Remove the first and last image

    # Use ThreadPoolExecutor to process images in parallel
    extracted_data = []
    with ThreadPoolExecutor() as executor:
        image_paths = [os.path.join(extract_to, filename) for filename in image_files if filename.endswith(('.png', '.jpg', '.jpeg', '.tiff', '.bmp'))]
        results = executor.map(process_image, image_paths)

    # Collect the results
    extracted_data.extend(list(results))

    # Create a DataFrame from the extracted data
    df = pd.DataFrame(extracted_data, columns=['Image Name', 'Extracted Text'])

    # Save the DataFrame to an Excel file in the same folder as the zip file
    output_excel = f"{zip_filename}.xlsx"
    output_path = os.path.join(subfolder_path, output_excel)  # Save to the same folder as the zip file
    df.to_excel(output_path, index=False, engine='openpyxl')

    print(f"Text extracted from images in zip file '{zip_filename}' and saved to {output_excel} in {subfolder_path}")

    # Check and delete the extracted images after processing
    for img_file in image_files:
        img_path = os.path.join(extract_to, img_file)
        if os.path.exists(img_path):
            os.remove(img_path)  # Only delete if the file exists

    # Delete the extracted folder after processing all images
    if os.path.exists(extract_to):
        shutil.rmtree(extract_to)  # Remove the entire folder

def process_folder(subfolder_path):
    # List all zip files in the subfolder
    zip_files = [f for f in sorted(os.listdir(subfolder_path)) if f.endswith('.zip')]

    for zip_file in zip_files:
        zip_file_path = os.path.join(subfolder_path, zip_file)
        process_zip_file(zip_file_path, subfolder_path)

# Main loop to go through all subfolders in the main folder (i.e., category folders)
for category_folder in sorted(os.listdir(main_folder)):
    category_folder_path = os.path.join(main_folder, category_folder)

    # Check if the path is a directory (i.e., a category folder)
    if os.path.isdir(category_folder_path):
        process_folder(category_folder_path)

print("Processing completed for all folders.")

Text extracted from images in zip file '4 agreements - Book pt6' and saved to 4 agreements - Book pt6.xlsx in /content/drive/MyDrive/StudyTok/Books
Text extracted from images in zip file 'Atomic Habits' and saved to Atomic Habits.xlsx in /content/drive/MyDrive/StudyTok/Books
Text extracted from images in zip file 'Deep Work' and saved to Deep Work.xlsx in /content/drive/MyDrive/StudyTok/Books
Text extracted from images in zip file 'How to become a straight A Student' and saved to How to become a straight A Student.xlsx in /content/drive/MyDrive/StudyTok/Books
Text extracted from images in zip file 'Mindset - Book pt5' and saved to Mindset - Book pt5.xlsx in /content/drive/MyDrive/StudyTok/Books
Text extracted from images in zip file 'Productivity books' and saved to Productivity books.xlsx in /content/drive/MyDrive/StudyTok/Books


/usr/local/lib/python3.10/dist-packages/torch/nn/modules/rnn.py:917: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at ../aten/src/ATen/native/cudnn/RNN.cpp:1424.)
  result = _VF.lstm(input, hx, self._flat_weights, self.bias, self.num_layers,


Text extracted from images in zip file 'So Good They Cant Ignore You' and saved to So Good They Cant Ignore You.xlsx in /content/drive/MyDrive/StudyTok/Books
Text extracted from images in zip file 'What to read' and saved to What to read.xlsx in /content/drive/MyDrive/StudyTok/Books
Text extracted from images in zip file 'Ask a tutor' and saved to Ask a tutor.xlsx in /content/drive/MyDrive/StudyTok/Conversion
Text extracted from images in zip file 'Tutoruu Conversion P5 Feature AI Comparison' and saved to Tutoruu Conversion P5 Feature AI Comparison.xlsx in /content/drive/MyDrive/StudyTok/Conversion
Text extracted from images in zip file 'Tutoruu Conversion P6 Feature Find Tutors' and saved to Tutoruu Conversion P6 Feature Find Tutors.xlsx in /content/drive/MyDrive/StudyTok/Conversion
Text extracted from images in zip file 'Tutoruu Conversion P8 Feature Free Sessions' and saved to Tutoruu Conversion P8 Feature Free Sessions.xlsx in /content/drive/MyDrive/StudyTok/Conversion
Text extract